In [ ]:
!pip install facenet-pytorch -q

In [ ]:
import torch
print(torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))

In [ ]:
from facenet_pytorch import MTCNN
print("MTCNN import successful")

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

from facenet_pytorch import MTCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mtcnn = MTCNN(keep_all=False, device=device)
print("Using device:", device)

In [ ]:
import os
print(os.path.exists("/kaggle/input"))
print(os.listdir("/kaggle/input"))

In [ ]:
import os

print(os.listdir("/kaggle/input/datasets"))

In [ ]:
base_path = "/kaggle/input/datasets"

for root, dirs, files in os.walk(base_path):
    print(root)
    if files:
        print("  sample files:", files[:3])

In [ ]:
train_real_dir = "/kaggle/input/datasets/nithesh0402/deepfake-and-real-images/Train/Real"
train_fake_dir = "/kaggle/input/datasets/nithesh0402/deepfake-and-real-images/Train/Fake"

val_real_dir = "/kaggle/input/datasets/nithesh0402/deepfake-and-real-images/Validation/Real"
val_fake_dir = "/kaggle/input/datasets/nithesh0402/deepfake-and-real-images/Validation/Fake"

test_real_dir = "/kaggle/input/datasets/nithesh0402/deepfake-and-real-images/Test/Real"
test_fake_dir = "/kaggle/input/datasets/nithesh0402/deepfake-and-real-images/Test/Fake"

print("Train Real:", len(os.listdir(train_real_dir)))
print("Train Fake:", len(os.listdir(train_fake_dir)))
print("Val Real:", len(os.listdir(val_real_dir)))
print("Val Fake:", len(os.listdir(val_fake_dir)))
print("Test Real:", len(os.listdir(test_real_dir)))
print("Test Fake:", len(os.listdir(test_fake_dir)))

In [ ]:
import pandas as pd
import os
import random

def make_df(real_dir, fake_dir, max_per_class=None):
    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir)]
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir)]

    if max_per_class is not None:
        real_files = random.sample(real_files, min(max_per_class, len(real_files)))
        fake_files = random.sample(fake_files, min(max_per_class, len(fake_files)))

    data = []
    for f in real_files:
        data.append([f, 0])   # 0 = real
    for f in fake_files:
        data.append([f, 1])   # 1 = fake

    df = pd.DataFrame(data, columns=["image_path", "label"])
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df = make_df(train_real_dir, train_fake_dir, max_per_class=8000)
val_df   = make_df(val_real_dir, val_fake_dir, max_per_class=2000)
test_df  = make_df(test_real_dir, test_fake_dir, max_per_class=2000)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))
train_df.head()

In [ ]:
import pandas as pd
import os
import random

def make_df(real_dir, fake_dir, max_per_class=None):
    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir)]
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir)]

    if max_per_class is not None:
        real_files = random.sample(real_files, min(max_per_class, len(real_files)))
        fake_files = random.sample(fake_files, min(max_per_class, len(fake_files)))

    data = []
    for f in real_files:
        data.append([f, 0])   # 0 = real
    for f in fake_files:
        data.append([f, 1])   # 1 = fake

    df = pd.DataFrame(data, columns=["image_path", "label"])
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df = make_df(train_real_dir, train_fake_dir, max_per_class=8000)
val_df   = make_df(val_real_dir, val_fake_dir, max_per_class=2000)
test_df  = make_df(test_real_dir, test_fake_dir, max_per_class=2000)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))
train_df.head()

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset
import cv2
import torch

class DeepfakeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, "image_path"]
        label = self.dataframe.loc[idx, "label"]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
from torch.utils.data import DataLoader

train_dataset = DeepfakeDataset(train_df, transform=train_transform)
val_dataset = DeepfakeDataset(val_df, transform=val_test_transform)
test_dataset = DeepfakeDataset(test_df, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
print("Labels:", labels[:10])

def imshow_tensor(img_tensor):
    img = img_tensor.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.axis("off")

plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    imshow_tensor(images[i].cpu())
    plt.title("Fake" if labels[i].item() == 1 else "Real")
plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch import nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

print("Using device:", device)

In [ ]:
from torch import optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
from sklearn.metrics import accuracy_score

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    return running_loss / len(loader), accuracy_score(all_labels, all_preds)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    return running_loss / len(loader), accuracy_score(all_labels, all_preds)

In [ ]:
num_epochs = 5
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_deepfake_resnet18.pth")
        print("Best model saved.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model.load_state_dict(torch.load("/kaggle/working/best_deepfake_resnet18.pth"))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Test Accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Real", "Fake"]))
print(confusion_matrix(all_labels, all_preds))

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": ["Real", "Fake"]
}, "/kaggle/working/deepfake_model_checkpoint.pth")

print("Checkpoint saved.")

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def predict_single_image(image_path, model, transform, device):
    model.eval()

    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    input_tensor = transform(image_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label_map = {0: "Real", 1: "Fake"}
    predicted_label = label_map[pred]
    confidence = probs[0][pred].item()

    plt.figure(figsize=(5, 5))
    plt.imshow(image_rgb)
    plt.title(f"Prediction: {predicted_label} | Confidence: {confidence:.4f}")
    plt.axis("off")
    plt.show()

    return predicted_label, confidence

In [ ]:
sample_real = test_df[test_df["label"] == 0]["image_path"].iloc[0]
predict_single_image(sample_real, model, val_test_transform, device)

In [ ]:
sample_fake = test_df[test_df["label"] == 1]["image_path"].iloc[0]
predict_single_image(sample_fake, model, val_test_transform, device)

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nithesh0402/deepfake-videos-dataset/DeepFake Videos Dataset.csv")
print(df.columns)
print(df.head())